# Evaluate RT-ICL with cross-validation
Run `01_preprocess.ipynb` first. This notebook performs 10-fold cross-validation with split seeds **0, 2, and 4**. Each query uses 10 reference compounds retrieved from its training folds.

In [1]:
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "rt_icl").is_dir() or not (ROOT / "data/raw").is_dir():
    raise RuntimeError("Open this notebook with the release folder as the working directory.")
RAW_DIR = ROOT / "data/raw"
DATA_DIR = ROOT / "data/processed"
RESULTS_DIR = ROOT / "results"

## Model and run settings
Supported choices:

| Model |
|---|
| `gemini-3-flash-preview` |
| `gpt-5.4-mini-2026-03-17` |
| `qwen/qwen3-235b-a22b-2507` |
| `openai/gpt-oss-120b` |


In [2]:
from rt_icl import (
    DEFAULT_CONFIG, DEFAULT_DATASETS, load_dataset, create_provider,
    prepare_cross_validation_requests, run_cross_validation,
)

DATASET_IDS = ["0004"]  # Use list(DEFAULT_DATASETS) for all 10 datasets.
SEEDS = DEFAULT_CONFIG.seeds  # (0, 2, 4)
PROVIDER = "gemini"
MODEL = "gemini-3-flash-preview"
MAX_CONCURRENT = 40

## Load data and preview a prompt

In [3]:
datasets = {dataset_id: load_dataset(dataset_id, DATA_DIR) for dataset_id in DATASET_IDS}
preview_data = datasets[DATASET_IDS[0]]
preview_requests = prepare_cross_validation_requests(
    preview_data.compounds, preview_data.lc_condition, seed=SEEDS[0],
)
print(preview_requests[0].prompt.user_content)
print("Planned API requests:", sum(len(data.compounds) for data in datasets.values()) * len(SEEDS))

You are an expert analytical chemist specializing in Liquid Chromatography–Mass Spectrometry (LC–MS).
Your task is to predict the retention time (RT) of a query compound under the specified target chromatographic system.
You are provided with the following information:
- The LC conditions of the target chromatographic system.
- A query compound whose RT is unknown.
- A set of structurally similar reference compounds along with their measured RTs.
- Relevant molecular identifiers and descriptors for the query and reference compounds. 

<Conditions>
LC mode: Reversed-Phase
Column: Thermo Scientific Hypersil GOLD (2.1 mm × 150 mm, 1.9 μm)
Mobile phase A: 0.1% formic acid in water (pH 3)
Mobile phase B: 0.1% formic acid in acetonitrile (pH 3)
Flow rate: 0.5 mL/min
Gradient:
  0 min (0 s): 100% A, 0% B
  2 min (120 s): 100% A, 0% B
  13 min (780 s): 0% A, 100% B
  15.5 min (930 s): 0% A, 100% B
  19 min (1140 s): 100% A, 0% B
</Conditions>

- Use the reference compounds and their measured R

## API keys
Copy `.env.example` to `.env` in the project folder if you do not already have one. Enter the API key for your selected `PROVIDER`: `GOOGLE_API_KEY`, `OPENAI_API_KEY`, or `OPENROUTER_API_KEY`. Then run this cell to load it.

In [4]:
import os
from dotenv import load_dotenv

load_dotenv(ROOT / ".env", override=True)

key_name = {"gemini": "GOOGLE_API_KEY", "openai": "OPENAI_API_KEY", "openrouter": "OPENROUTER_API_KEY"}[PROVIDER]
if not os.environ.get(key_name):
    raise ValueError(f"Set {key_name} in the project .env file.")

## Execute
The next cell makes API requests. Existing result files are checked before starting the run; use a new output location to repeat it.

In [5]:
import re
from rt_icl.evaluation import check_result_conflicts

provider = create_provider(PROVIDER, model=MODEL)
model_tag = re.sub(r"[^A-Za-z0-9._-]+", "_", provider.model).strip("._-") or "model"
runs = [
    (dataset_id, seed, RESULTS_DIR / "dataset" / dataset_id / PROVIDER / model_tag / f"seed{seed}")
    for dataset_id in DATASET_IDS for seed in SEEDS
]
for _, _, output_dir in runs:
    check_result_conflicts(output_dir)

results = []
for dataset_id, seed, output_dir in runs:
    data = datasets[dataset_id]
    result = await run_cross_validation(
        data.compounds, data.lc_condition, provider,
        dataset_id=dataset_id, seed=seed, max_concurrent=MAX_CONCURRENT,
        output_dir=output_dir,
    )
    results.append(result)
    print(dataset_id, "seed", seed, result.metrics["overall"])

Direct use of automatic function calling (AFC) in AsyncModels.generate_content is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message. Similarly, direct use of AFC in AsyncModels.generate_content_stream is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message_stream.


0004 seed 0 {'total': 110, 'parsed': 110, 'parse_failures': 0, 'parse_rate': 1.0, 'api_errors': 0, 'labeled': 110, 'evaluated': 110, 'mae': 45.834545454545456, 'median_absolute_error': 15.5, 'mape_percent': 31.510491600181307, 'r2': 0.6830288340649263}
0004 seed 2 {'total': 110, 'parsed': 110, 'parse_failures': 0, 'parse_rate': 1.0, 'api_errors': 0, 'labeled': 110, 'evaluated': 110, 'mae': 45.83181818181818, 'median_absolute_error': 18.0, 'mape_percent': 32.64206405125468, 'r2': 0.6889586917801205}
0004 seed 4 {'total': 110, 'parsed': 110, 'parse_failures': 0, 'parse_rate': 1.0, 'api_errors': 0, 'labeled': 110, 'evaluated': 110, 'mae': 40.49272727272727, 'median_absolute_error': 16.25, 'mape_percent': 27.442002896275593, 'r2': 0.7114163401270586}


## Review metrics and saved outputs

In [6]:
summary = [
    {"dataset": result.config["dataset_id"], "seed": result.config["split"]["seed"],
     **result.metrics["overall"], "output_dir": str(result.output_dir)}
    for result in results
]
display(summary)

[{'dataset': '0004',
  'seed': 0,
  'total': 110,
  'parsed': 110,
  'parse_failures': 0,
  'parse_rate': 1.0,
  'api_errors': 0,
  'labeled': 110,
  'evaluated': 110,
  'mae': 45.834545454545456,
  'median_absolute_error': 15.5,
  'mape_percent': 31.510491600181307,
  'r2': 0.6830288340649263,
  'output_dir': '/home/dm3/woojae/00_RT/RT-ICL/results/dataset/0004/gemini/gemini-3-flash-preview/seed0'},
 {'dataset': '0004',
  'seed': 2,
  'total': 110,
  'parsed': 110,
  'parse_failures': 0,
  'parse_rate': 1.0,
  'api_errors': 0,
  'labeled': 110,
  'evaluated': 110,
  'mae': 45.83181818181818,
  'median_absolute_error': 18.0,
  'mape_percent': 32.64206405125468,
  'r2': 0.6889586917801205,
  'output_dir': '/home/dm3/woojae/00_RT/RT-ICL/results/dataset/0004/gemini/gemini-3-flash-preview/seed2'},
 {'dataset': '0004',
  'seed': 4,
  'total': 110,
  'parsed': 110,
  'parse_failures': 0,
  'parse_rate': 1.0,
  'api_errors': 0,
  'labeled': 110,
  'evaluated': 110,
  'mae': 40.49272727272727,


Each run saves `run_metadata.json`, `metrics.json`, and `predictions.csv`. Metrics include MAE (s), MedAE (s), MAPE (%), and R².